In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score

# --- 1. ADIM: Doğrudan İnternetten Veriyi Çekme ---
# Drive ile uğraşmadan IBM'in orijinal ve temiz dosyasını çekiyoruz.
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

# --- 2. ADIM: Temizlik ---
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(inplace=True)
df.drop('customerID', axis=1, inplace=True) # Sadece gereksiz kimlikleri atıyoruz

# --- 3. ADIM: Hedef (y) ve Özellik (X) Ayrımı ---
X = df.drop('Churn', axis=1)
y = df['Churn'].map({'Yes': 1, 'No': 0})

X_encoded = pd.get_dummies(X, drop_first=True)

# --- 4. ADIM: Veriyi Bölme (80 Eğitim - 10 Validasyon - 10 Test) ---
X_train, X_temp, y_train, y_temp = train_test_split(X_encoded, y, test_size=0.2, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

# --- 5. ADIM: SMOTE (Sınıf Dengesizliği Çözümü) ---
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# --- 6. ADIM: YENİ MODEL - XGBoost (Hiperparametre Ayarlı) ---
# Modeli %78'in üzerine çıkarmak için ince ayarlar (Tuning) yapıyoruz:
xgb_model = XGBClassifier(
    random_state=42,
    learning_rate=0.05,    # 1. Ayar: Modelin öğrenme hızı (daha yavaş ama daha sağlam ve dikkatli öğrenir)
    max_depth=4,           # 2. Ayar: Ağaçların derinliği (Ezberlemeyi -Overfitting- önler)
    n_estimators=200,      # 3. Ayar: Toplam ağaç sayısı
    subsample=0.8,         # 4. Ayar: Her ağaçta verinin %80'ini kullanarak farklı senaryolar çalıştırır
    colsample_bytree=0.8,  # 5. Ayar: Sütunların da %80'ini seçerek kopya çekmesini zorlaştırır
    use_label_encoder=False,
    eval_metric='logloss'
)

# Modeli eğitiyoruz
xgb_model.fit(X_train_smote, y_train_smote)

# Validasyon (Ara Sınav) setiyle test ediyoruz
y_pred = xgb_model.predict(X_val)

# --- SONUÇLAR ---
print("\n--- XGBOOST MODEL BAŞARI KARNESİ (VALIDASYON SETİ) ---")
print(classification_report(y_val, y_pred))

accuracy = accuracy_score(y_val, y_pred) * 100
print(f"Genel Doğruluk Oranı (Accuracy): %{accuracy:.2f}")

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [11:46:08] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



--- XGBOOST MODEL BAŞARI KARNESİ (VALIDASYON SETİ) ---
              precision    recall  f1-score   support

           0       0.86      0.81      0.84       516
           1       0.55      0.64      0.59       187

    accuracy                           0.77       703
   macro avg       0.71      0.72      0.71       703
weighted avg       0.78      0.77      0.77       703

Genel Doğruluk Oranı (Accuracy): %76.53
